# processed 파일 import

In [ ]:
import pandas as pd
from google.colab import files
uploaded = files.upload()

Saving 그녀가 죽었다_processed.csv to 그녀가 죽었다_processed.csv


In [ ]:
title = '그녀가 죽었다'
df = pd.read_csv(f'{title}_processed.csv')
df.head()

,Index,processed,별점,감상평
0,1,사이코 라이 구도 숭맹숭 초반 긴장감 마지막 이상하다 풀다 버리다 소재 재밌다 쌓다...,2.0,걍 사이코 대 또라이 구도로 가지\n한 명이 너무 밍숭맹숭 해졌다\n꼭 초반의 긴장...
1,2,처음 느껴지다 소재 반짝임 지키다,3.0,처음부터 느껴진 소재의 반짝임을 꽤나 지켜냈다할까.
2,3,결말 궁금하다 마지막 보기 완료 피해자 가해자 마음 착하다,3.0,24/10/18\n결말이 궁금하여 마지막 부분 보기 완료. 피해자인척 말고 언제든 ...
3,4,빼다 가볍다 보기 좋다,3.0,뇌빼고 가볍게 보기 좋음
4,5,관객 응시 연출 좋다,3.0,관객을 응시하는 연출이 좋았다


In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# 데이터프레임 예시 (이미 존재하는 데이터프레임 df)
# df = pd.read_csv('your_file.csv')  # 데이터프레임 로드하는 부분

# 별점 라벨 인코딩을 위해 값 변환
df['별점'] = df['별점'].astype(float)

# 라벨 인코더 초기화
label_encoder = LabelEncoder()

# 별점 라벨 인코딩 (0.5 -> 1, 1.0 -> 2, ..., 5.0 -> 10)
df['별점_encoded'] = label_encoder.fit_transform(df['별점']) + 1  # 라벨 인코딩 결과가 0부터 시작하므로 +1

df['별점_encoded'].unique()


array([ 4,  6,  2,  5,  1,  8,  9,  7,  3, 10])

# 로지스틱 회귀

In [ ]:
#!pip install xlsxwriter


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.9/159.9 kB 4.6 MB/s eta 0:00:00


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from google.colab import files

# 데이터프레임 정렬 (Index를 기준으로 오름차순 정렬)
df = df.sort_values(by='Index', ascending=True)

# NaN 값을 빈 문자열로 대체
df['processed'] = df['processed'].fillna('')

# train/test set 나누기 (최근 40%를 test set으로 사용)
split_index = int(len(df) * 0.6)
train_df = df.iloc[:split_index]
test_df = df.iloc[split_index:]

# TF-IDF 벡터화 (processed 텍스트)
vectorizer = TfidfVectorizer()
X_train = vectorizer.fit_transform(train_df['processed'])
X_test = vectorizer.transform(test_df['processed'])

# 종속 변수 (별점_encoded)
y_train = train_df['별점_encoded']
y_test = test_df['별점_encoded']

# 로지스틱 회귀 모델 훈련
model = LogisticRegression(max_iter=1000, multi_class='multinomial', solver='lbfgs')
model.fit(X_train, y_train)

# 예측
y_pred = model.predict(X_test)

# 결과 평가
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred, output_dict=True)

# classification report를 데이터프레임으로 변환
report_df = pd.DataFrame(report).transpose()

# y_pred에 0.5를 곱해 '예측 별점' 생성
df_test = test_df.copy()  # test 데이터 복사
df_test['예측 별점'] = y_pred * 0.5  # 예측된 값에 0.5 곱해서 실제 별점으로 변환

# 'gap' 열 생성 (실제 별점 - 예측 별점)
df_test['gap'] = df_test['별점'] - df_test['예측 별점']

# df_final에서 '별점_encoded' 열 제거 및 필요한 열 순서 지정
df_final = df_test.drop(columns=['별점_encoded'])
df_final = df_final[['Index', '감상평', '별점', '예측 별점', 'gap']]

# xlsx 파일명 생성
file_name = f"{title}_로지스틱_평점분류예측.xlsx"

# 여러 시트를 가진 Excel 파일로 저장
with pd.ExcelWriter(file_name, engine='xlsxwriter') as writer:
    df_final.to_excel(writer, sheet_name='Prediction Results', index=False)  # 첫 번째 시트에 저장
    report_df.to_excel(writer, sheet_name='Classification Report')  # 두 번째 시트에 저장

# 저장한 파일 다운로드
files.download(file_name)


/usr/local/lib/python3.10/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Decision Tree 분류

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, accuracy_score
from google.colab import files

# 데이터프레임 정렬 (Index를 기준으로 오름차순 정렬)
df = df.sort_values(by='Index', ascending=True)

# NaN 값을 빈 문자열로 대체
df['processed'] = df['processed'].fillna('')

# train/test set 나누기 (최근 40%를 test set으로 사용)
split_index = int(len(df) * 0.6)
train_df = df.iloc[:split_index]
test_df = df.iloc[split_index:]

# TF-IDF 벡터화 (processed 텍스트)
vectorizer = TfidfVectorizer()
X_train = vectorizer.fit_transform(train_df['processed'])
X_test = vectorizer.transform(test_df['processed'])

# 종속 변수 (별점_encoded)
y_train = train_df['별점_encoded']
y_test = test_df['별점_encoded']

# Decision Tree 모델 훈련
model = DecisionTreeClassifier(random_state=42)
model.fit(X_train, y_train)

# 예측
y_pred = model.predict(X_test)

# 결과 평가
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred, output_dict=True)

# classification report를 데이터프레임으로 변환
report_df = pd.DataFrame(report).transpose()

# y_pred에 0.5를 곱해 '예측 별점' 생성
df_test = test_df.copy()  # test 데이터 복사
df_test['예측 별점'] = y_pred * 0.5  # 예측된 값에 0.5 곱해서 실제 별점으로 변환

# 'gap' 열 생성 (실제 별점 - 예측 별점)
df_test['gap'] = df_test['별점'] - df_test['예측 별점']

# df_final에서 '별점_encoded' 열 제거 및 필요한 열 순서 지정
df_final = df_test.drop(columns=['별점_encoded'])
df_final = df_final[['Index', '감상평', '별점', '예측 별점', 'gap']]

# xlsx 파일명 생성
file_name = f"{title}_DecisionTree_평점분류예측.xlsx"

# 여러 시트를 가진 Excel 파일로 저장
with pd.ExcelWriter(file_name, engine='xlsxwriter') as writer:
    df_final.to_excel(writer, sheet_name='Prediction Results', index=False)  # 첫 번째 시트에 저장
    report_df.to_excel(writer, sheet_name='Classification Report')  # 두 번째 시트에 저장

# 저장한 파일 다운로드
files.download(file_name)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# SVM

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score
from google.colab import files

# 데이터프레임 정렬 (Index를 기준으로 오름차순 정렬)
df = df.sort_values(by='Index', ascending=True)

# NaN 값을 빈 문자열로 대체
df['processed'] = df['processed'].fillna('')

# train/test set 나누기 (최근 40%를 test set으로 사용)
split_index = int(len(df) * 0.6)
train_df = df.iloc[:split_index]
test_df = df.iloc[split_index:]

# TF-IDF 벡터화 (processed 텍스트)
vectorizer = TfidfVectorizer()
X_train = vectorizer.fit_transform(train_df['processed'])
X_test = vectorizer.transform(test_df['processed'])

# 종속 변수 (별점_encoded)
y_train = train_df['별점_encoded']
y_test = test_df['별점_encoded']

# SVM 모델 훈련
model = SVC(kernel='linear', C=1, random_state=42)  # 선형 커널 사용
model.fit(X_train, y_train)

# 예측
y_pred = model.predict(X_test)

# 결과 평가
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred, output_dict=True)

# classification report를 데이터프레임으로 변환
report_df = pd.DataFrame(report).transpose()

# y_pred에 0.5를 곱해 '예측 별점' 생성
df_test = test_df.copy()  # test 데이터 복사
df_test['예측 별점'] = y_pred * 0.5  # 예측된 값에 0.5 곱해서 실제 별점으로 변환

# 'gap' 열 생성 (실제 별점 - 예측 별점)
df_test['gap'] = df_test['별점'] - df_test['예측 별점']

# df_final에서 '별점_encoded' 열 제거 및 필요한 열 순서 지정
df_final = df_test.drop(columns=['별점_encoded'])
df_final = df_final[['Index', '감상평', '별점', '예측 별점', 'gap']]

# xlsx 파일명 생성
file_name = f"{title}_SVM_평점분류예측.xlsx"

# 여러 시트를 가진 Excel 파일로 저장
with pd.ExcelWriter(file_name, engine='xlsxwriter') as writer:
    df_final.to_excel(writer, sheet_name='Prediction Results', index=False)  # 첫 번째 시트에 저장
    report_df.to_excel(writer, sheet_name='Classification Report')  # 두 번째 시트에 저장

# 저장한 파일 다운로드
files.download(file_name)


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>